# 02 — Preprocessing

Missing-value analysis, duplicate checks, invalid-value cleanup, and
outlier detection (IQR method) for `application_train.csv`. This mirrors
the logic in `utils/preprocessing.py`, which the Streamlit pages call
directly — this notebook shows the same steps worked through explicitly.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append("..")

from utils.preprocessing import (
    clean_application_data, dataset_overview, column_profile, duplicate_summary,
    missing_value_summary, missing_bucket, suggest_treatment, iqr_bounds, outlier_summary,
)

app = pd.read_csv("../data/application_train.csv")
app.shape

(1000, 122)

## Step 2 — Missing values

In [2]:
missing = app.isnull().sum()
missing_pct = (missing / len(app) * 100).round(2)
missing_df = pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})
missing_df = missing_df[missing_df["missing_count"] > 0].sort_values("missing_count", ascending=False)
missing_df.head(20)

,missing_count,missing_pct
COMMONAREA_AVG,697,69.7
COMMONAREA_MODE,697,69.7
COMMONAREA_MEDI,697,69.7
NONLIVINGAPARTMENTS_AVG,689,68.9
NONLIVINGAPARTMENTS_MODE,689,68.9
NONLIVINGAPARTMENTS_MEDI,689,68.9
LIVINGAPARTMENTS_AVG,682,68.2
LIVINGAPARTMENTS_MODE,682,68.2
LIVINGAPARTMENTS_MEDI,682,68.2
FONDKAPREMONT_MODE,680,68.0


In [3]:
summary = missing_value_summary(app)
summary["bucket"] = summary["Missing %"].apply(missing_bucket)
summary["bucket"].value_counts()

bucket
40-60% Missing    32
60%+ Missing      17
0-5% Missing       7
5-20% Missing      6
20-40% Missing     2
Name: count, dtype: int64

In [4]:
summary["suggested_treatment"] = summary.apply(lambda r: suggest_treatment(r["Missing %"], r["Data Type"]), axis=1)
summary[["Column", "Missing %", "suggested_treatment"]].head(15)

,Column,Missing %,suggested_treatment
0,COMMONAREA_AVG,69.7,Drop column — too sparse to reliably impute
1,COMMONAREA_MODE,69.7,Drop column — too sparse to reliably impute
2,COMMONAREA_MEDI,69.7,Drop column — too sparse to reliably impute
3,NONLIVINGAPARTMENTS_AVG,68.9,Drop column — too sparse to reliably impute
4,NONLIVINGAPARTMENTS_MODE,68.9,Drop column — too sparse to reliably impute
5,NONLIVINGAPARTMENTS_MEDI,68.9,Drop column — too sparse to reliably impute
6,LIVINGAPARTMENTS_AVG,68.2,Drop column — too sparse to reliably impute
7,LIVINGAPARTMENTS_MODE,68.2,Drop column — too sparse to reliably impute
8,LIVINGAPARTMENTS_MEDI,68.2,Drop column — too sparse to reliably impute
9,FONDKAPREMONT_MODE,68.0,Drop column — too sparse to reliably impute


**Treatment logic (see `utils/preprocessing.suggest_treatment`):**
- `>60%` missing → drop column (too sparse to reliably impute)
- `40-60%` missing → retain + missing indicator (missingness itself may be informative)
- `<40%` missing, numeric → fill with median (robust to skew)
- `<40%` missing, categorical → fill with mode / 'Unknown'

## Step 3 — Duplicate analysis

In [5]:
print("Full-row duplicates:", app.duplicated().sum())
print("SK_ID_CURR unique:", app["SK_ID_CURR"].is_unique)
print("Duplicate SK_ID_CURR values:", app["SK_ID_CURR"].duplicated().sum())
duplicate_summary(app)

Full-row duplicates: 0
SK_ID_CURR unique: True
Duplicate SK_ID_CURR values: 0


{'full_row_duplicates': 0, 'duplicate_ids': 0, 'id_is_unique': True}

## Step 4 — Data type correction & Step 5 — invalid values

DAYS_EMPLOYED contains an anomalous placeholder (365243) mostly for pensioners / not-currently-employed applicants.

In [6]:
print("DAYS_EMPLOYED == 365243 count:", (app["DAYS_EMPLOYED"] == 365243).sum())
print("This is", round((app['DAYS_EMPLOYED'] == 365243).mean() * 100, 1), "% of the sample")
app["DAYS_EMPLOYED"].describe()

DAYS_EMPLOYED == 365243 count: 158
This is 15.8 % of the sample


count      1000.000000
mean      55733.906000
std      134159.108526
min      -15632.000000
25%       -2724.000000
50%       -1267.000000
75%        -304.000000
max      365243.000000
Name: DAYS_EMPLOYED, dtype: float64

In [7]:
print("CODE_GENDER unique values:", app["CODE_GENDER"].unique())
print("Any non-positive income/credit/annuity values?")
for col in ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]:
    print(f"  {col}: {(app[col] <= 0).sum()} non-positive values")

CODE_GENDER unique values: <ArrowStringArray>
['M', 'F']
Length: 2, dtype: str
Any non-positive income/credit/annuity values?
  AMT_INCOME_TOTAL: 0 non-positive values
  AMT_CREDIT: 0 non-positive values
  AMT_ANNUITY: 0 non-positive values
  AMT_GOODS_PRICE: 0 non-positive values


In [8]:
app_clean = clean_application_data(app)
print("DAYS_EMPLOYED anomaly after cleaning:", (app_clean["DAYS_EMPLOYED"] == 365243).sum())
print("Now missing instead:", app_clean["DAYS_EMPLOYED"].isna().sum() - app["DAYS_EMPLOYED"].isna().sum())

DAYS_EMPLOYED anomaly after cleaning: 0
Now missing instead: 158


## Step 6 — Outlier analysis (IQR method)

In [9]:
VARIABLES = ["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE",
             "DAYS_BIRTH", "DAYS_EMPLOYED", "CNT_CHILDREN", "CNT_FAM_MEMBERS"]
outliers = outlier_summary(app_clean, VARIABLES)
outliers

,Column,Q1,Q3,IQR,Lower Bound,Upper Bound,Outlier Count,Outlier %
0,AMT_INCOME_TOTAL,112500.00,202500.00,90000.00,-22500.00,337500.00,50,5.00
1,AMT_CREDIT,273584.25,808650.00,535065.75,-529014.38,1611248.62,17,1.70
2,AMT_ANNUITY,16603.88,34103.25,17499.38,-9645.19,60352.31,26,2.60
3,AMT_GOODS_PRICE,238500.00,679500.00,441000.00,-423000.00,1341000.00,43,4.30
4,DAYS_BIRTH,-19229.00,-12326.50,6902.50,-29582.75,-1972.75,0,0.00
5,DAYS_EMPLOYED,-3140.75,-735.00,2405.75,-6749.38,2873.62,56,6.65
6,CNT_CHILDREN,0.00,1.00,1.00,-1.50,2.50,13,1.30
7,CNT_FAM_MEMBERS,2.00,2.25,0.25,1.62,2.62,487,48.70


**Interpretation:** we do not automatically remove flagged outliers. A high income + large
loan combination is a true extreme customer (keep); a value of exactly 0 or 1 is more likely a
data-entry issue (already handled by `clean_application_data`, which treats non-positive
money values as missing); very large family/children counts get a business-rule cap rather
than blanket IQR removal, since count variables behave differently from skewed money fields.

## Dataset overview after cleaning

In [10]:
dataset_overview(app_clean)

{'rows': 1000,
 'columns': 122,
 'numeric_columns': 106,
 'categorical_columns': 16,
 'missing_cells': 29554,
 'duplicate_rows': 0,
 'memory_usage_mb': np.float64(1.0592832565307617),
 'unique_customers': 1000}

## Next notebook
03_feature_engineering.ipynb builds AGE_YEARS, EMPLOYMENT_YEARS, affordability ratios, bands, and the customer-level aggregates from the related tables.